In [1]:
import datetime
import os
import sys
from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorRetriever, HybridRetriever, HybridCypherRetriever

from openai import AzureOpenAI
from pprint import pprint
import numpy as np
import yaml
import os
import pandas as pd
from databricks import sql

sys.path.append("../")

from src.utils.utils import load_config

load_dotenv()

True

In [2]:
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],
)

LLM_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]
EMBEDDING_MODEL = os.environ["AZURE_OPENAI_EMBEDDING_MODEL"]
EMBEDDING_DIMENSION = 3072

VECTOR_INDEX_NAME = "table_vector_index"
FULLTEXT_INDEX_NAME = "table_fulltext_index"
TABLE_EMBEDDING_PROPERTY_NAME = "embedding"

config = {
    'TABLE_VECTOR_INDEX_NAME': VECTOR_INDEX_NAME,
    'TABLE_EMBEDDING_PROPERTY_NAME': TABLE_EMBEDDING_PROPERTY_NAME,
    'EMBEDDING_DIMENSIONS': EMBEDDING_DIMENSION,
    'TABLE_FULLTEXT_INDEX_NAME': FULLTEXT_INDEX_NAME,
    'TABLE_TEXT_PROPERTIES': ['name', 'description'] # List of properties to include in the fulltext index
}

In [3]:
# --- Extract Databricks connection info from environment ---
host = os.environ.get("DATABRICKS_HOST")
token = os.environ.get("DATABRICKS_TOKEN")
warehouse_id = os.environ.get("DATABRICKS_WAREHOUSE_ID")
catalog = os.environ.get("DATABRICKS_CATALOG")
schema = os.environ.get("DATABRICKS_SCHEMA")

# --- Assertions to make sure the variables exist ---
assert host, "DATABRICKS_HOST environment variable is not set"
assert token, "DATABRICKS_TOKEN environment variable is not set"
assert warehouse_id, "DATABRICKS_WAREHOUSE_ID environment variable is not set"
assert catalog, "DATABRICKS_CATALOG environment variable is not set"
assert schema, "DATABRICKS_SCHEMA environment variable is not set"

# --- Connect to Databricks SQL ---
conn = sql.connect(
    server_hostname=host,
    http_path=f"/sql/1.0/warehouses/{warehouse_id}",
    access_token=token,
)

# --- Info about the data ---
print(f"Connected to Databricks SQL at {host}")
print(f"Catalog: {catalog}, Schema: {schema}")
print("Metadata about tables and columns in this schema can now be queried for building a knowledge graph or data catalog.")

Connected to Databricks SQL at dbc-01fbcd3c-2696.cloud.databricks.com
Catalog: business_intelligence_systems, Schema: 01_bronce
Metadata about tables and columns in this schema can now be queried for building a knowledge graph or data catalog.


In [4]:
# --- Step 1: Query columns and tables ---
columns_query = f"""
SELECT
    t.table_schema,
    t.table_name,
    t.comment AS table_description,
    c.column_name,
    c.comment AS column_description,
    c.data_type AS data_type
FROM system.information_schema.tables t
JOIN system.information_schema.columns c
    ON t.table_catalog = c.table_catalog
    AND t.table_schema = c.table_schema
    AND t.table_name = c.table_name
WHERE t.table_catalog = '{catalog}'
  AND t.table_schema = '{schema}'
ORDER BY t.table_name, c.ordinal_position
"""
df_columns = pd.read_sql(columns_query, conn)

# --- Step 2: Query key_column_usage for PK/FK info ---
kcu_query = f"""
SELECT *
FROM {catalog}.information_schema.key_column_usage
WHERE table_schema = '{schema}'
"""
df_keys = pd.read_sql(kcu_query, conn)

# --- Step 3: Function to determine PK/FK ---
def get_key_type(row):
    matches = df_keys[
        (df_keys["table_name"] == row["table_name"]) &
        (df_keys["column_name"] == row["column_name"])
    ]
    if matches.empty:
        return ""

    key_types = []
    for _, k in matches.iterrows():
        if pd.isna(k.get("position_in_unique_constraint")):
            key_types.append("PK")
        else:
            # Try to find referenced column
            ref = df_keys[df_keys["constraint_name"] == k["position_in_unique_constraint"]]
            if not ref.empty:
                ref_table = ref.iloc[0]["table_name"]
                ref_column = ref.iloc[0]["column_name"]
                key_types.append(f"FK -> {ref_table}.{ref_column}")
            else:
                key_types.append("FK")
    return ", ".join(key_types)

df_columns["key_type"] = df_columns.apply(get_key_type, axis=1)

# --- Step 4: Sample 5 values for each column ---
def get_sample_values(row, n=5):
    table = row["table_name"]
    column = row["column_name"]
    try:
        sample_query = f"""
        SELECT {column}
        FROM {catalog}.{schema}.{table}
        LIMIT {n}
        """
        df_sample = pd.read_sql(sample_query, conn)
        # Convert to comma-separated string
        return ", ".join(df_sample[column].astype(str).tolist())
    except Exception as e:
        return f"Error: {e}"

df_columns["sample_values"] = df_columns.apply(get_sample_values, axis=1)

/var/folders/g_/mf1cyg4x7mx1t773_fkxk7xm0000gn/T/ipykernel_68979/3152471760.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_columns = pd.read_sql(columns_query, conn)
/var/folders/g_/mf1cyg4x7mx1t773_fkxk7xm0000gn/T/ipykernel_68979/3152471760.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_keys = pd.read_sql(kcu_query, conn)
/var/folders/g_/mf1cyg4x7mx1t773_fkxk7xm0000gn/T/ipykernel_68979/3152471760.py:65: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(sample_query, conn)
/var/folders/

In [5]:
df_columns.head()

,table_schema,table_name,table_description,column_name,column_description,data_type,key_type,sample_values
0,01_bronce,categories,The table contains information about product c...,CategoryID,Unique identifier for each product category.,LONG,PK,"Error: int() argument must be a string, a byte..."
1,01_bronce,categories,The table contains information about product c...,CategoryName,Name of the product category.,STRING,,"Beverages, Condiments, Confections, Dairy Prod..."
2,01_bronce,categories,The table contains information about product c...,Description,Detailed information about the product category,STRING,,"Soft drinks, coffees, teas, beers, and ales, S..."
3,01_bronce,categories,The table contains information about product c...,Picture,Binary data for the image associated with the ...,BINARY,,Error: 'utf-8' codec can't decode byte 0xff in...
4,01_bronce,customers,The table contains information about customers...,CustomerID,NaN,STRING,PK,"ALFKI, ANATR, ANTON, AROUT, BERGS"


In [6]:
# Get credentials from environment variables
neo4j_url = os.getenv("NEO4J_BOLT_URL", "bolt://localhost:7687")
neo4j_user = os.getenv("NEO4J_USERNAME",  "neo4j")
neo4j_password = os.getenv("NEO4J_PASSWORD", "password")

# Connect to Neo4j
driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_password))

with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    value = result.single()["test"]
    if value == 1:
        print("✅ Neo4j connection successful!")
    else:
        print("⚠️ Neo4j connection test failed.")


✅ Neo4j connection successful!


In [7]:
# Flush the graph database
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

print("Graph database flushed successfully.")

Graph database flushed successfully.


In [8]:
embedding_model = os.environ.get("EMBEDDING_MODEL", EMBEDDING_MODEL)

# --- Step 1: Prepare distinct tables ---
distinct_tables = df_columns[['table_name', 'table_schema', 'table_description']].drop_duplicates().reset_index(drop=True)

# --- Step 2: Prepare column texts for embeddings ---
column_texts = [
    f"Table: {row['table_name']}, Column: {row['column_name']}, Samples: {row['sample_values']}"
    for _, row in df_columns.iterrows()
]

# --- Step 3: Get column embeddings ---
column_embeddings_response = client.embeddings.create(
    input=column_texts,
    model=embedding_model
)
column_embeddings = [item.embedding for item in column_embeddings_response.data]

# --- Step 4: Get table embeddings based on table description ---
table_texts = [
    row['table_description'] if pd.notna(row['table_description']) else f"Table: {row['table_name']}"
    for _, row in distinct_tables.iterrows()
]

table_embeddings_response = client.embeddings.create(
    input=table_texts,
    model=embedding_model
)
table_embeddings = [item.embedding for item in table_embeddings_response.data]

# --- Step 5: Upsert Table and Column nodes in Neo4j ---
with driver.session() as session:

    for idx, row in df_columns.iterrows():
        table_name = row['table_name']
        col_name = table_name + "." + row['column_name']
        column_embedding_vector = column_embeddings[idx]

        # Safely get table description
        table_description = row.get('table_description')
        if pd.isna(table_description) or table_description is None:
            table_description = ""

        # Upsert Table node (embedding from description)
        table_idx = distinct_tables[distinct_tables['table_name'] == table_name].index[0]
        table_embedding_vector = table_embeddings[table_idx]

        session.run(
            """
            MERGE (t:Table {name: $table_name, schema: $schema})
            SET t.description = $description,
                t.embedding = $embedding
            """,
            table_name=table_name,
            schema=row['table_schema'],
            description=table_description,
            embedding=table_embedding_vector
        )

        # Upsert Column node
        session.run(
            """
            MERGE (c:Column {name: $column_name})
            SET c.description = $column_description,
                c.key_type = $key_type,
                c.sample_values = $sample_values,
                c.embedding = $embedding
            """,
            column_name=col_name,
            column_description=row['column_description'],
            key_type=row['key_type'],
            sample_values=row['sample_values'],
            embedding=column_embedding_vector
        )

        # Create relationship
        session.run(
            """
            MATCH (t:Table {name: $table_name})
            MATCH (c:Column {name: $column_name})
            MERGE (t)-[:HAS_COLUMN]->(c)
            """,
            table_name=table_name,
            column_name=col_name
        )

print("✅ Tables and columns created with embeddings. Table embeddings now based on table descriptions.")

✅ Tables and columns created with embeddings. Table embeddings now based on table descriptions.


In [9]:
# Extract PK and FK columns
# Identify primary keys and foreign keys
pk_columns = df_columns[df_columns['key_type'] == 'PK'].copy()
fk_columns = df_columns[df_columns['key_type'] == 'FK'].copy()
pk_columns.head()

,table_schema,table_name,table_description,column_name,column_description,data_type,key_type,sample_values
0,01_bronce,categories,The table contains information about product c...,CategoryID,Unique identifier for each product category.,LONG,PK,"Error: int() argument must be a string, a byte..."
4,01_bronce,customers,The table contains information about customers...,CustomerID,NaN,STRING,PK,"ALFKI, ANATR, ANTON, AROUT, BERGS"
15,01_bronce,employees,The table contains information about employees...,EmployeeID,Unique identifier for each employee,LONG,PK,"Error: int() argument must be a string, a byte..."
40,01_bronce,orders,The table captures information related to cust...,OrderID,"A unique identifier assigned to each order, en...",LONG,PK,"Error: int() argument must be a string, a byte..."
54,01_bronce,products,The table contains information about products ...,ProductID,A unique identifier assigned to each product i...,LONG,PK,"Error: int() argument must be a string, a byte..."


In [10]:
fk_columns = df_columns[df_columns['key_type'] == 'FK'].copy()
fk_columns.head()

,table_schema,table_name,table_description,column_name,column_description,data_type,key_type,sample_values
41,01_bronce,orders,The table captures information related to cust...,CustomerID,Identifier for the customer who placed the ord...,STRING,FK,"VINET, TOMSP, HANAR, VICTE, SUPRD"
42,01_bronce,orders,The table captures information related to cust...,EmployeeID,Identifier for the employee responsible for pr...,LONG,FK,"Error: int() argument must be a string, a byte..."
56,01_bronce,products,The table contains information about products ...,SupplierID,A unique identifier for the supplier of the pr...,LONG,FK,"Error: int() argument must be a string, a byte..."
57,01_bronce,products,The table contains information about products ...,CategoryID,A unique identifier that categorizes the produ...,LONG,FK,"Error: int() argument must be a string, a byte..."


In [11]:
fk_to_pk_mapping = {}

for _, fk_row in fk_columns.iterrows():
    fk_col_name = fk_row['column_name']
    # Look for a PK column with the same name
    match = pk_columns[pk_columns['column_name'] == fk_col_name]
    
    if not match.empty:
        # take the first match
        pk_row = match.iloc[0]
        fk_full_name = f"{fk_row['table_name']}.{fk_col_name}"
        pk_full_name = f"{pk_row['table_name']}.{pk_row['column_name']}"
        fk_to_pk_mapping[fk_full_name] = pk_full_name
    else:
        print(f"⚠️ Could not find PK match for FK column: {fk_row['table_name']}.{fk_col_name}")

In [12]:
with driver.session() as session:
    for fk_col, pk_col in fk_to_pk_mapping.items():
        session.run(
            """
            MATCH (fk:Column {name: $fk_col})
            MATCH (pk:Column {name: $pk_col})
            MERGE (fk)-[:REFERENCES]->(pk)
            """,
            fk_col=fk_col,
            pk_col=pk_col
        )

In [13]:
with driver.session() as session:
    # Get all existing indexes
    result = session.run("SHOW INDEXES")
    indexes = [record["name"] for record in result]

    # Drop each index
    for idx in indexes:
        session.run(f"DROP INDEX {idx} IF EXISTS")
        print(f"Dropped index: {idx}")

Dropped index: table_fulltext_index
Dropped index: table_vector_index


In [14]:
def create_indexes(vector_index_name, 
                   embedding_property, 
                   vector_dimensions,
                   fulltext_index_name,
                   fulltext_properties):
    """
    Creates a Neo4j vector index and a full-text index for the Table nodes.
    """
    # Vector index query
    vector_query = f"""
        CREATE VECTOR INDEX {vector_index_name}
        FOR (t:Table)
        ON (t.{embedding_property})
        OPTIONS {{
            indexConfig: {{
                `vector.dimensions`: {vector_dimensions},
                `vector.similarity_function`: "cosine"
            }}
        }}
    """

    # Full-text index query
    properties_str = ", ".join([f"t.{p}" for p in fulltext_properties])
    fulltext_query = f"""
    CREATE FULLTEXT INDEX {fulltext_index_name}
    FOR (t:Table)
    ON EACH [{properties_str}]
    """

    with driver.session() as session:
        # Drop existing indexes if they exist
        session.run(f"DROP INDEX {vector_index_name} IF EXISTS")
        session.run(f"DROP INDEX {fulltext_index_name} IF EXISTS")

        # Create new indexes
        session.run(vector_query)
        session.run(fulltext_query)

        print(f"Vector index `{vector_index_name}` created on `{embedding_property}` ({vector_dimensions} dims).")
        print(f"Full-text index `{fulltext_index_name}` created on properties: {fulltext_properties}.")

In [15]:
create_indexes(
    vector_index_name=config['TABLE_VECTOR_INDEX_NAME'],
    embedding_property=config['TABLE_EMBEDDING_PROPERTY_NAME'],
    vector_dimensions=config['EMBEDDING_DIMENSIONS'],
    fulltext_index_name=config['TABLE_FULLTEXT_INDEX_NAME'],
    fulltext_properties=config['TABLE_TEXT_PROPERTIES']
)

Vector index `table_vector_index` created on `embedding` (3072 dims).
Full-text index `table_fulltext_index` created on properties: ['name', 'description'].


In [16]:
# Setup Cypher query for HybridCypherRetriever (full-text only fallback)
retrieval_query = """
MATCH (t:Table)
OPTIONAL MATCH (t)-[:HAS_COLUMN]->(c:Column)

RETURN 
  t.name AS table_name,
  t.schema AS table_schema,
  t.description AS table_description,
  [col IN COLLECT(DISTINCT c) 
    WHERE col.name IS NOT NULL | 
      {
        column_name: col.name,
        description: COALESCE(col.description, "No description available"),
        data_type: COALESCE(col.data_type, "Unknown")
      }
  ] AS columns
LIMIT $top_k
"""

In [17]:
class OpenAIEmbedder:
    def __init__(self, model=EMBEDDING_MODEL, api_key=None):
        self.model = model
        if api_key:
            AzureOpenAI.api_key = api_key
    def embed_query(self, text):
        response = client.embeddings.create(
            input=text,
            model=self.model
        )
        return response.data[0].embedding

In [18]:

retriever = VectorRetriever(
    driver,
    index_name=config['TABLE_VECTOR_INDEX_NAME'],
    embedder=OpenAIEmbedder(),
    return_properties=["name", "schema", "description"],
)

query_text = "Wie viele Bestellungen hat Mitarbeiter Sara bearbeitet?"
retriever_result = retriever.search(query_text=query_text, top_k=3)

output_lines = []

for item in retriever_result.items:
    raw_text = item.content
    if raw_text.startswith("<Record "):
        raw_text = raw_text[len("<Record "):]
    if raw_text.endswith(">"):
        raw_text = raw_text[:-1]

    # Normalize extra whitespace
    raw_text = " ".join(raw_text.split())

    # Add the cleaned line to the list
    output_lines.append(raw_text)
    output_lines.append("\n")

retrieved_contents = "\n".join(output_lines)
print(retrieved_contents)

{'name': 'orders', 'schema': '01_bronce', 'description': 'The table captures information related to customer orders. It includes details such as customer information, order dates, shipping details, and freight costs. This data can be used to analyze order fulfillment processes, track shipping efficiency, and assess customer purchasing patterns.'}


{'name': 'order_details', 'schema': '01_bronce', 'description': 'The table contains detailed information about customer orders. It includes data regarding the order ID, product details, pricing, quantity ordered, and any discounts applied. This table can be used for analyzing sales performance, calculating revenue, and identifying trends in purchasing behaviors.'}


{'name': 'shippers', 'schema': '01_bronce', 'description': 'The table contains information about shippers associated with our operations. It includes details such as the shipper ID, company name, and contact phone number. This data can be used for logistics management, vendor ass

In [19]:
retriever = HybridRetriever(
    driver,
    vector_index_name=config['TABLE_VECTOR_INDEX_NAME'],
    fulltext_index_name=config['TABLE_FULLTEXT_INDEX_NAME'],
    embedder=OpenAIEmbedder(),
    return_properties=["name"],
)

query_text="wie viele bestellungen hat mitarbeiter sara bearbeitet"
retriever_result = retriever.search(query_text=query_text, top_k=3)

output_lines = []

for item in retriever_result.items:
    raw_text = item.content
    if raw_text.startswith("<Record "):
        raw_text = raw_text[len("<Record "):]
    if raw_text.endswith(">"):
        raw_text = raw_text[:-1]

    # Normalize extra whitespace
    raw_text = " ".join(raw_text.split())

    # Add the cleaned line to the list
    output_lines.append(raw_text)
    output_lines.append("\n")

retrieved_contents = "\n".join(output_lines)
print(retrieved_contents)

{'name': 'orders'}


{'name': 'order_details'}


{'name': 'shippers'}




In [20]:
# Setup Cypher query for HybridCypherRetriever (full-text only fallback)
retrieval_query = """
WITH node AS t, score
OPTIONAL MATCH (t)-[:HAS_COLUMN]->(c:Column)

RETURN 
  t.name AS table_name,
  t.schema AS table_schema,
  score AS retrieval_score,
  t.description AS table_description,
  [col IN COLLECT(DISTINCT c) 
    WHERE col.name IS NOT NULL | 
      {
        column_name: col.name,
        description: COALESCE(col.description, "No description available"),
        data_type: COALESCE(col.data_type, "Unknown")
      }
  ] AS columns
ORDER BY retrieval_score DESC
LIMIT $top_k
"""

retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=config['TABLE_VECTOR_INDEX_NAME'],
    fulltext_index_name=config['TABLE_FULLTEXT_INDEX_NAME'],
    embedder=OpenAIEmbedder(),
    #return_properties=["name", "schema", "description"],
    retrieval_query=retrieval_query,
)


query_text="wie viele bestellungen hat mitarbeiter sara bearbeitet"
retriever_result = retriever.search(query_text=query_text, top_k=3)

output_lines = []

for item in retriever_result.items:
    raw_text = item.content
    if raw_text.startswith("<Record "):
        raw_text = raw_text[len("<Record "):]
    if raw_text.endswith(">"):
        raw_text = raw_text[:-1]

    # Normalize extra whitespace
    raw_text = " ".join(raw_text.split())

    # Add the cleaned line to the list
    output_lines.append(raw_text)
    output_lines.append("\n")

retrieved_contents = "\n".join(output_lines)
print(retrieved_contents)

table_name='orders' table_schema='01_bronce' retrieval_score=1.0 table_description='The table captures information related to customer orders. It includes details such as customer information, order dates, shipping details, and freight costs. This data can be used to analyze order fulfillment processes, track shipping efficiency, and assess customer purchasing patterns.' columns=[{'column_name': 'orders.ShipCountry', 'data_type': 'Unknown', 'description': 'The country where the order is being shipped, important for international shipping analysis and compliance.'}, {'column_name': 'orders.ShipPostalCode', 'data_type': 'Unknown', 'description': 'The postal code for the shipping address, aiding in accurate delivery and logistics planning.'}, {'column_name': 'orders.ShipRegion', 'data_type': 'Unknown', 'description': 'The region or state associated with the shipping address, providing additional geographic context for shipping data.'}, {'column_name': 'orders.ShipCity', 'data_type': 'Unkn

In [21]:
def generate_search_string(query_text: str, client, LLM_MODEL) -> str:
    """
    Splits a natural language query into independent topics for retrieval,
    provides a meaningful search direction for each topic, and returns
    a single concatenated search string.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert in analyzing queries and creating meaningful search directions "
                "for SQL database retrieval. For each topic, provide a concise instruction describing "
                "what to look for in the database related to that topic."
                "Do not include any additional text or explanations."
                "Do not fabricate any information that is not explicitly mentioned in the query. Focus on extracting clear search directions based solely on the query content."
            )
        },
        {
            "role": "user",
            "content": (
                f"Split this query into independent non-technical topics and provide a meaningful search direction for each:\n'{query_text}'\n"
                "Output as a JSON array of objects with keys 'topic' and 'search_direction'."
            )
        }
    ]
    
    response = client.chat.completions.create(
        messages=messages,
        model=LLM_MODEL
    )
    
    topics_json = response.choices[0].message.content
    try:
        topics = json.loads(topics_json)
    except:
        # fallback: naive split if JSON parsing fails
        topics = [{"topic": t, "search_direction": f"search for relevant records about '{t}'"} for t in query_text.split()]
    
    # Combine all search directions into one string
    search_directions = [t["search_direction"] for t in topics]
    return "; ".join(search_directions), len(topics)


# Example usage:
query_text = "wie viele bestellungen hat mitarbeiter Sara bearbeitet"
long_search_string, num_topics = generate_search_string(query_text, client, LLM_MODEL)
print(long_search_string)
print(f"Number of search topics: {num_topics}")

search for relevant records about 'wie'; search for relevant records about 'viele'; search for relevant records about 'bestellungen'; search for relevant records about 'hat'; search for relevant records about 'mitarbeiter'; search for relevant records about 'Sara'; search for relevant records about 'bearbeitet'
Number of search topics: 7


In [22]:
# Adjust top_k based on number of topics for complexer queries, take 3 results as base, limit to 5
top_k = min(3 + num_topics - 1, 5)

retriever_result = retriever.search(query_text=long_search_string, top_k=top_k)

output_lines = []

for item in retriever_result.items:
    raw_text = item.content
    if raw_text.startswith("<Record "):
        raw_text = raw_text[len("<Record "):]
    if raw_text.endswith(">"):
        raw_text = raw_text[:-1]

    # Normalize extra whitespace
    raw_text = " ".join(raw_text.split())

    # Add the cleaned line to the list
    output_lines.append(raw_text)
    output_lines.append("\n")

retrieved_contents = "\n".join(output_lines)
print(retrieved_contents)

table_name='employees' table_schema='01_bronce' retrieval_score=1.0 table_description='The table contains information about employees, including personal details, job titles, and contact information. It can be used for HR management, employee data analysis, and tracking personnel-related metrics. Useful insights may include employee demographics, tenure, and reporting structure.' columns=[{'column_name': 'employees.PhotoPath', 'data_type': 'Unknown', 'description': "Stores the file path to the employee's photo, facilitating access and display in employee directories and profiles."}, {'column_name': 'employees.ReportsTo', 'data_type': 'Unknown', 'description': "Indicates the identifier of the employee's manager or supervisor, which helps in understanding the reporting structure within the organization."}, {'column_name': 'employees.Notes', 'data_type': 'Unknown', 'description': 'Provides a space for any additional remarks or comments regarding the employee, which may include performance

In [23]:
def generate_initial_sql(retrieved_contents: str, query_text: str):
    """
    STEP 1: Generates an initial SQL query from the user query and retrieved knowledge.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert in generating SQL queries from natural language queries, especially in the banking industry."
            )
        },
        {
            "role": "user",
            "content": f"""
            Here is the top ranked retrieved relevant data records (based on the
            user's query) from a knowledge graph containing semantic information of a SQL database:
            
            {retrieved_contents}
            
            ---
            Using the above retrieved information of the database, generate a syntactically correct SQL query to help
            find the answer for the user's query: {query_text}.
            
            Do not use any other information or knowledge other than the retrieved information above. Within this retrieved
            information, make sure you rank and use the most appropriate and relevant tables and columns when generating the
            SQL code to answer the user query.
            
            For more context, 'Concept' refers to the linkage of various columns to a single concept or entity.
            It is also useful for you to understand the primary key and foreign key linkages.
            
            Output the final SQL query as plain text (and nothing else).
            """
        }
    ]
    response = client.chat.completions.create(
        messages=messages,
        model=LLM_MODEL
    )
    sql_output = response.choices[0].message.content
    return sql_output.strip()

In [24]:
# OPTIONAL: Set up double checker of SQL query to validate
def double_check_and_refine_sql(initial_sql: str, query_text: str):
    """
    STEP 2: Double-check the initial SQL to see if it addresses the user's question.
    If needed, refine it. Return the refined SQL (or the same one if it's already good).
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a SQL expert. You will receive an initial SQL query and decide if it truly addresses the user's question. "
                "If needed, refine or fix it. Otherwise, confirm it's correct."
            )
        },
        {
            "role": "user",
            "content": f"""
            The user's query is: "{query_text}"
            The initially generated SQL is:
            
            {initial_sql}
            
            Check if:
            1. The SQL is syntactically correct.
            2. The SQL will answer the user's question accurately (based on the user query).
            3. The SQL only uses columns/tables suggested by the database info (no out-of-scope info).
            
            If something is missing or incorrect, refine/fix the SQL. Otherwise, confirm it's already correct.
            
            Output the final, refined SQL as plain text (and nothing else).
            """
        }
    ]
    response = client.chat.completions.create(
        messages=messages,
        model=LLM_MODEL
    )
    refined_sql = response.choices[0].message.content
    return refined_sql.strip()

In [25]:
def run_sql_generation_pipeline(query_text: str, retrieved_contents: str):
    """
    Orchestrates the 2-step pipeline.
    1) Generate initial SQL
    2) Double-check/refine
    """
    initial_sql = generate_initial_sql(retrieved_contents, query_text)
    refined_sql = double_check_and_refine_sql(initial_sql, query_text)
    print("=== USER QUERY ===")
    print(query_text, "\n")
    print("=== RETRIEVED CONTENTS ===")
    print(retrieved_contents, "\n")
    print("=== INITIAL SQL ===")
    print(initial_sql, "\n")
    print("=== FINAL SQL ===")
    print(refined_sql, "\n")

    return refined_sql

In [26]:
query_text="was wurde am letzten verfügbaren tag am meisten bestellt?"
long_search_string, num_topics = generate_search_string(query_text, client, LLM_MODEL)

top_k = min(3 + num_topics - 1, 5)
retriever_result = retriever.search(query_text=long_search_string, top_k=top_k)

output_lines = []

for item in retriever_result.items:
    raw_text = item.content
    if raw_text.startswith("<Record "):
        raw_text = raw_text[len("<Record "):]
    if raw_text.endswith(">"):
        raw_text = raw_text[:-1]

    # Normalize extra whitespace
    raw_text = " ".join(raw_text.split())

    # Add the cleaned line to the list
    output_lines.append(raw_text)
    output_lines.append("\n")

retrieved_contents = "\n".join(output_lines)

final_sql_query = run_sql_generation_pipeline(query_text, retrieved_contents)

print("==== RESULT===")

with sql.connect(
    server_hostname=host,
    http_path=f"/sql/1.0/warehouses/{warehouse_id}",
    access_token=token
) as connection:

    with connection.cursor() as cursor:
        # Use named parameters in extended format style
        cursor.execute(f"USE CATALOG {catalog};")
        cursor.execute(final_sql_query)
        result_arrow = cursor.fetchall_arrow()
        df = result_arrow.to_pandas()  # Pandas handles None/NaN safely
        df = df.fillna(0)  # replace NULLs with 0
        print(df)

=== USER QUERY ===
was wurde am letzten verfügbaren tag am meisten bestellt? 

=== RETRIEVED CONTENTS ===
table_name='orders' table_schema='01_bronce' retrieval_score=1.0 table_description='The table captures information related to customer orders. It includes details such as customer information, order dates, shipping details, and freight costs. This data can be used to analyze order fulfillment processes, track shipping efficiency, and assess customer purchasing patterns.' columns=[{'column_name': 'orders.ShipCountry', 'data_type': 'Unknown', 'description': 'The country where the order is being shipped, important for international shipping analysis and compliance.'}, {'column_name': 'orders.ShipPostalCode', 'data_type': 'Unknown', 'description': 'The postal code for the shipping address, aiding in accurate delivery and logistics planning.'}, {'column_name': 'orders.ShipRegion', 'data_type': 'Unknown', 'description': 'The region or state associated with the shipping address, providing